In [ ]:
import pandas as pd
import numpy as np

df=pd.read_json('regex_output.ndjson',lines=True)
df.head(2)

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by
0,mac,Jul 1 09:00:55 calvisitor-10-105-160-95 kerne...,iothunderboltswitch<<NUM>>(0x0)::listenercallb...,INFO,kernel,2026-07-01 09:00:55,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.0,Unclassified
1,mac,Jul 1 09:01:05 calvisitor-10-105-160-95 com.a...,thermal pressure state: <NUM> memory pressure ...,WARN,com.apple.CDScheduler,2026-07-01 09:01:05,"{'host': 'calvisitor-10-105-160-95', 'pid': '43'}",None,0.0,Unclassified


In [ ]:
df.classified_by.value_counts()

,count
classified_by,
Unclassified,8670
regex,1330


In [ ]:
from sentence_transformers import SentenceTransformer

class BertEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed(self, texts):
        """
        texts: List[str]
        returns: numpy array of embeddings
        """
        return self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class BertEngine:
    def __init__(self, embedder, similarity_threshold=0.80):
        self.embedder = embedder
        self.similarity_threshold = similarity_threshold

        # Memory of known embeddings
        self.known_embeddings = []
        self.known_labels = []

    def add_known_examples(self, texts, labels):
        """
        texts: List[str]
        labels: List[str]
        """
        embeddings = self.embedder.embed(texts)
        self.known_embeddings.extend(embeddings)
        self.known_labels.extend(labels)

    def classify(self, text):
        """
        Classify a single Unclassified log
        """
        if not self.known_embeddings:
            return {
                "label": None,
                "confidence": 0.0,
                "classified_by": "bert"
            }

        query_embedding = self.embedder.embed([text])[0]

        sims = cosine_similarity(
            [query_embedding],
            self.known_embeddings
        )[0]

        best_idx = int(np.argmax(sims))
        best_score = float(sims[best_idx])

        if best_score >= self.similarity_threshold:
            return {
                "label": self.known_labels[best_idx],
                "confidence": best_score,
                "classified_by": "bert"
            }

        return {
            "label": None,
            "confidence": best_score,
            "classified_by": "bert"
        }


In [ ]:
import json

INPUT = "./regex_output.ndjson"
OUTPUT = "./bert_output.ndjson"

embedder = BertEmbedder()
engine = BertEngine(embedder)

# Step 1: Load known labeled logs
known_texts = []
known_labels = []

with open(INPUT) as f:
    for line in f:
        log = json.loads(line)
        if log["label"] and log["classified_by"] == "regex":
            known_texts.append(log["clean_message"])
            known_labels.append(log["label"])

engine.add_known_examples(known_texts, known_labels)

# Step 2: Classify unclassified logs
with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        log = json.loads(line)

        if log["classified_by"] == "Unclassified":
            result = engine.classify(log["clean_message"])
            log.update(result)

        fout.write(json.dumps(log) + "\n")

print("Phase 2 completed.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Phase 2 completed.


In [ ]:
import pandas as pd
df=pd.read_json('bert_output.ndjson',lines=True)
df.head()

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by
0,mac,Jul 1 09:00:55 calvisitor-10-105-160-95 kerne...,iothunderboltswitch<<NUM>>(0x0)::listenercallb...,INFO,kernel,2026-07-01 09:00:55,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.250470,bert
1,mac,Jul 1 09:01:05 calvisitor-10-105-160-95 com.a...,thermal pressure state: <NUM> memory pressure ...,WARN,com.apple.CDScheduler,2026-07-01 09:01:05,"{'host': 'calvisitor-10-105-160-95', 'pid': '43'}",None,0.372277,bert
2,mac,Jul 1 09:01:06 calvisitor-10-105-160-95 QQ[10...,fa||url||taskid[<NUM>] dealloc,INFO,QQ,2026-07-01 09:01:06,"{'host': 'calvisitor-10-105-160-95', 'pid': '1...",None,0.339929,bert
3,mac,Jul 1 09:02:26 calvisitor-10-105-160-95 kerne...,arpt: <NUM>.<NUM>: airport_brcm43xx::syncpower...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.396106,bert
4,mac,Jul 1 09:02:26 authorMacBook-Pro kernel[0]: A...,arpt: <NUM>.<NUM>: airport_brcm43xx::platformw...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'authorMacBook-Pro', 'pid': '0', 'com...",None,0.394218,bert


In [ ]:
df.classified_by.value_counts()

,count
classified_by,
bert,8670
regex,1330


In [ ]:
df[df['confidence'] >= .75].shape

(1926, 10)

In [ ]:
from sentence_transformers import SentenceTransformer

class BertEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def encode(self, texts):
        return self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )
import joblib
import numpy as np

class MLEngine:
    def __init__(self, model_path, threshold=0.80):
        self.model = joblib.load(model_path)
        self.embedder = BertEmbedder()
        self.threshold = threshold

    def classify(self, text):
        embedding = self.embedder.encode([text])
        probs = self.model.predict_proba(embedding)[0]

        best_idx = int(np.argmax(probs))
        best_prob = float(probs[best_idx])
        best_label = self.model.classes_[best_idx]

        if best_prob >= self.threshold:
            return {
                "label": best_label,
                "confidence": best_prob,
                "classified_by": "ml"
            }

        return {
            "label": None,
            "confidence": best_prob,
            "classified_by": "ml"
        }


In [ ]:
import json
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

INPUT = "./regex_output.ndjson"
MODEL_PATH = "./log_classifier.joblib"

texts, labels = [], []

with open(INPUT) as f:
    for line in f:
        log = json.loads(line)
        if (log["label"]!= "" ) and log["classified_by"] != "regex":
            texts.append(log["clean_message"])
            labels.append(log["label"])

print(f"[INFO] Training samples: {len(texts)}")

embedder = BertEmbedder()
X = embedder.encode(texts)
y = labels

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y
)

y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

clf.fit(X_train, y_train)

print(classification_report(y_val, clf.predict(X_val)))

joblib.dump(clf, MODEL_PATH)
print("[DONE] Model trained and saved")


[INFO] Training samples: 8670


TypeError: '<' not supported between instances of 'NoneType' and 'NoneType'

In [ ]:
import json
import numpy as np
import joblib
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# Assuming embedder is already initialized:
# embedder = BertEmbedder()

le = LabelEncoder()

INPUT = "./regex_output.ndjson"
MODEL_PATH = "./log_classifier.joblib"

texts, labels = [], []

print("[INFO] Loading data...")

with open(INPUT) as f:
    for line in f:
        try:
            log = json.loads(line)

            # Safe extraction
            label_val = str(log.get("label") or "").strip()
            message = log.get("clean_message", "").strip()

            # --- FIX IS HERE ---
            # We removed 'and log.get("classified_by") != "regex"'
            # Now we accept ANY valid label, regardless of source.
            if label_val != "" and message != "":
                texts.append(message)
                labels.append(label_val)

        except json.JSONDecodeError:
            continue

# 1. Validation
label_counts = Counter(labels)
print(f"[INFO] Samples loaded: {len(texts)}")
print(f"[INFO] Label distribution: {dict(label_counts)}")

if len(label_counts) < 2:
    raise ValueError(f"Still not enough classes. Found: {list(label_counts.keys())}")

# 2. Vectorization
print("[INFO] Encoding texts...")
X = embedder.encode(texts)
y = np.array(labels)

# 3. Stratified Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Encoding Targets
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# 5. Training
print("[INFO] Training model...")
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)
clf.fit(X_train, y_train_encoded)

# 6. Evaluation
print("\n--- Classification Report ---")
preds = clf.predict(X_val)
print(classification_report(y_val_encoded, preds, target_names=le.classes_))

# 7. Save
# joblib.dump({"model": clf, "encoder": le}, MODEL_PATH)
# print(f"[DONE] Saved model to {MODEL_PATH}")

[INFO] Loading data...
[INFO] Samples loaded: 1330
[INFO] Label distribution: {'AUTH_FAILURE': 490, 'UNKNOWN_USER_LOGIN': 117, 'HDFS_BLOCK_STORED': 314, 'HDFS_BLOCK_RECEIVED': 294, 'HDFS_BLOCK_ALLOCATED': 115}
[INFO] Encoding texts...
[INFO] Training model...

--- Classification Report ---
                      precision    recall  f1-score   support

        AUTH_FAILURE       1.00      1.00      1.00        98
HDFS_BLOCK_ALLOCATED       1.00      1.00      1.00        23
 HDFS_BLOCK_RECEIVED       1.00      1.00      1.00        59
   HDFS_BLOCK_STORED       1.00      1.00      1.00        63
  UNKNOWN_USER_LOGIN       1.00      1.00      1.00        23

            accuracy                           1.00       266
           macro avg       1.00      1.00      1.00       266
        weighted avg       1.00      1.00      1.00       266

[DONE] Saved model to ./log_classifier.joblib


In [ ]:
import joblib
import numpy as np
from embedder import BertEmbedder

class MLEngine:
    def __init__(self, model_path, threshold=0.80):
        self.model = joblib.load(model_path)
        self.embedder = BertEmbedder()
        self.threshold = threshold

    def classify(self, text):
        embedding = self.embedder.encode([text])
        probs = self.model.predict_proba(embedding)[0]

        best_idx = int(np.argmax(probs))
        best_prob = float(probs[best_idx])
        best_label = self.model.classes_[best_idx]

        if best_prob >= self.threshold:
            return {
                "label": best_label,
                "confidence": best_prob,
                "classified_by": "ml"
            }

        return {
            "label": None,
            "confidence": best_prob,
            "classified_by": "ml"
        }


In [ ]:
import json
from ml.ml_engine import MLEngine

INPUT = "regex_output.ndjson"
OUTPUT = "bert_output.ndjson"

engine = MLEngine("ml/log_classifier.joblib")

with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        log = json.loads(line)

        if log["classified_by"] == "Unclassified":
            result = engine.classify(log["clean_message"])
            log.update(result)

        fout.write(json.dumps(log) + "\n")

print("[DONE] Phase 2 ML inference complete")


In [ ]:
df[df.label=='APACHE_WORKER_INIT']

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by


In [ ]:
REGEX_RULES = [
    # ================= Security =================
    {
        "level": "ERROR",
        "pattern": r"authentication failure",
        "confidence": 0.95,
        "sources": ["linux"]
    },
    {
        "level": "WARNING",
        "pattern": r"check pass; user unknown",
        "confidence": 0.96,
        "sources": ["linux"]
    },

    # ================= Core Dumps =================
    {
        "level": "CRITICAL",
        "pattern": r"generating core\.\d+",
        "confidence": 0.98,
        "sources": ["bgl"]
    },

    # ================= HDFS =================
    {
        "level": "INFO",
        "pattern": r"packetresponder \d+ for block blk_.* terminating",
        "confidence": 0.90,
        "sources": ["hdfs"]
    },
    {
        "level": "INFO",
        "pattern": r"block\* namesystem\.addstoredblock",
        "confidence": 0.88,
        "sources": ["hdfs"]
    },
    {
        "level": "INFO",
        "pattern": r"received block blk_",
        "confidence": 0.87,
        "sources": ["hdfs"]
    },
    {
        "level": "INFO",
        "pattern": r"block\* namesystem\.allocateblock",
        "confidence": 0.87,
        "sources": ["hdfs"]
    },
]


In [ ]:
import re


class RegexEngine:
    def __init__(self):
        # Compile regex patterns once (fast + production-safe)
        self.rules = [
            {
                **rule,
                "compiled": re.compile(rule["pattern"], re.IGNORECASE)
            }
            for rule in REGEX_RULES
        ]

    def classify(self, log: dict) -> dict:
        """
        Phase 1: Regex-based severity classification.

        Returns:
            {
                "level": <LEVEL or None>,
                "confidence": <float>,
                "classified_by": "regex" | "Unclassified"
            }
        """
        text = log.get("clean_message", "")
        source = log.get("source")

        for rule in self.rules:
            if source not in rule["sources"]:
                continue

            if rule["compiled"].search(text):
                return {
                    "level": rule["level"],
                    "confidence": rule["confidence"],
                    "classified_by": "regex"
                }

        # Explicitly unresolved — hand off to Phase 2
        return {
            "level": None,
            "confidence": 0.0,
            "classified_by": "Unclassified"
        }


In [ ]:
import json

INPUT_PATH = "./unified_logs.ndjson"
OUTPUT_PATH = "./regex_output.ndjson"

regex_engine = RegexEngine()

total_logs = 0
regex_matched = 0

with open(INPUT_PATH, "r") as fin, open(OUTPUT_PATH, "w") as fout:
    for line in fin:
        log = json.loads(line)
        total_logs += 1

        # Phase 1: Regex severity classification
        result = regex_engine.classify(log)

        # Always write explicit fields (no ambiguity downstream)
        log["level"] = result.get("level")
        log["confidence"] = result.get("confidence", 0.0)
        log["classified_by"] = result.get("classified_by")

        if log["classified_by"] == "regex":
            regex_matched += 1

        fout.write(json.dumps(log) + "\n")

print(
    f"[PHASE 1 COMPLETE] "
    f"Regex classified {regex_matched}/{total_logs} logs "
    f"({regex_matched / max(total_logs, 1):.2%})"
)


[PHASE 1 COMPLETE] Regex classified 1330/10000 logs (13.30%)


In [ ]:
df=pd.read_json('regex_output.ndjson',lines=True)

In [ ]:
df.head()

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,level,confidence,classified_by
0,mac,Jul 1 09:00:55 calvisitor-10-105-160-95 kerne...,iothunderboltswitch<<NUM>>(0x0)::listenercallb...,INFO,kernel,2026-07-01 09:00:55,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",NaN,None,0.0,Unclassified
1,mac,Jul 1 09:01:05 calvisitor-10-105-160-95 com.a...,thermal pressure state: <NUM> memory pressure ...,WARN,com.apple.CDScheduler,2026-07-01 09:01:05,"{'host': 'calvisitor-10-105-160-95', 'pid': '43'}",NaN,None,0.0,Unclassified
2,mac,Jul 1 09:01:06 calvisitor-10-105-160-95 QQ[10...,fa||url||taskid[<NUM>] dealloc,INFO,QQ,2026-07-01 09:01:06,"{'host': 'calvisitor-10-105-160-95', 'pid': '1...",NaN,None,0.0,Unclassified
3,mac,Jul 1 09:02:26 calvisitor-10-105-160-95 kerne...,arpt: <NUM>.<NUM>: airport_brcm43xx::syncpower...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",NaN,None,0.0,Unclassified
4,mac,Jul 1 09:02:26 authorMacBook-Pro kernel[0]: A...,arpt: <NUM>.<NUM>: airport_brcm43xx::platformw...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'authorMacBook-Pro', 'pid': '0', 'com...",NaN,None,0.0,Unclassified


In [ ]:
# phase2/eval_split.py
from sklearn.model_selection import train_test_split

def split_data(X, y, test_size=0.2, seed=42):
    return train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=seed
    )


In [ ]:
# phase2/evaluate.py
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

def evaluate(model, label_encoder, X_val, y_val):
    y_true = label_encoder.transform(y_val)
    y_pred = model.predict(X_val)

    report = classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True
    )

    cm = confusion_matrix(y_true, y_pred)

    return report, cm


In [ ]:
# phase2/train_phase2.py (UPDATED)
import joblib
import json

DATA_PATH = "./regex_output.ndjson"

texts, levels = load_training_data(DATA_PATH)

embedder = BertEmbedder()
X = embedder.encode(texts)

X_train, X_val, y_train, y_val = split_data(X, levels)

model, label_encoder = train_classifier(X_train, y_train)

report, cm = evaluate(model, label_encoder, X_val, y_val)

# Save artifacts
joblib.dump(model, "./phase2_lr.pkl")
joblib.dump(label_encoder, "./phase2_label_encoder.pkl")

with open("models/phase2_eval.json", "w") as f:
    json.dump(report, f, indent=2)

print("[PHASE 2] Evaluation Summary")
for level, metrics in report.items():
    if level in label_encoder.classes_:
        print(
            f"{level:10s} | "
            f"P={metrics['precision']:.2f} "
            f"R={metrics['recall']:.2f} "
            f"F1={metrics['f1-score']:.2f}"
        )


AttributeError: 'BertEmbedder' object has no attribute 'encode'

In [ ]:
import json

def load_phase2_training_data(path):
    texts = []
    levels = []

    with open(path, "r") as f:
        for line in f:
            log = json.loads(line)

            # ✅ TRAIN ONLY ON UNCLASSIFIED LOGS
            if log.get("classified_by") != "Unclassified":
                continue

            # ✅ MUST HAVE GROUND-TRUTH SEVERITY
            if log.get("log_level") is None:
                continue

            texts.append(log["clean_message"])
            levels.append(log["log_level"])

    return texts, levels


In [ ]:
texts, levels = load_phase2_training_data(
    "./regex_output.ndjson"
)

# BERT embeddings
X = embedder.encode(texts)

# Encode labels
y = label_encoder.fit_transform(levels)

# Train model
model.fit(X, y)


AttributeError: 'BertEmbedder' object has no attribute 'encode'

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
import os

os.makedirs("model", exist_ok=True)

# ----------------------------------
# 1. Load structured logs
# ----------------------------------
df = pd.read_json("regex_output.ndjson",lines=True)

# ----------------------------------
# 2. TRAINING DATA SELECTION (CRITICAL)
# ----------------------------------
df = df[
    (df["classified_by"] == "Unclassified") &
    (df["log_level"].notna())
]

print(f"Training samples: {len(df)}")

texts = df["clean_message"].astype(str).tolist()
labels = df["log_level"].tolist()

# ----------------------------------
# 3. Encode log levels
# ----------------------------------
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

joblib.dump(label_encoder, "model/label_encoder.pkl")

# ----------------------------------
# 4. Load BERT
# ----------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()

# ----------------------------------
# 5. Convert logs → BERT embeddings
# ----------------------------------
def bert_embeddings(texts, batch_size=16):
    vectors = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            )

            outputs = bert(**inputs)

            cls_vec = outputs.last_hidden_state[:, 0, :]
            vectors.append(cls_vec.cpu().numpy())

    return np.vstack(vectors)

X = bert_embeddings(texts)

# ----------------------------------
# 6. Train / Test split
# ----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ----------------------------------
# 7. Train ML classifier
# ----------------------------------
clf = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    class_weight="balanced"
)

clf.fit(X_train, y_train)

# ----------------------------------
# 8. Evaluation
# ----------------------------------
y_pred = clf.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

# ----------------------------------
# 9. Save model
# ----------------------------------
joblib.dump(clf, "model/log_level_clf.pkl")

print("✅ Phase-2 trained ONLY on Unclassified logs")


Training samples: 8670


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

              precision    recall  f1-score   support

       ERROR       0.97      0.98      0.98       236
        INFO       1.00      0.99      1.00      1138
      NOTICE       1.00      1.00      1.00       281
     UNKNOWN       1.00      1.00      1.00        50
        WARN       1.00      1.00      1.00        29

    accuracy                           0.99      1734
   macro avg       0.99      1.00      0.99      1734
weighted avg       0.99      0.99      0.99      1734

✅ Phase-2 trained ONLY on Unclassified logs


In [ ]:
import torch
import joblib
import numpy as np
from transformers import BertTokenizer, BertModel

# -------------------------------
# Load models
# -------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()

clf = joblib.load("model/log_level_clf.pkl")
label_encoder = joblib.load("model/label_encoder.pkl")

# -------------------------------
# BERT embedding function
# -------------------------------
def embed(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert(**inputs)

    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

# -------------------------------
# Predict log level
# -------------------------------
def predict_log_level(log_message, threshold=0.7):
    vector = embed(log_message)

    probs = clf.predict_proba(vector)[0]
    max_prob = np.max(probs)
    label_idx = np.argmax(probs)

    predicted_level = label_encoder.inverse_transform([label_idx])[0]

    if max_prob < threshold:
        return {
            "log_level": "Uncertain",
            "confidence": float(max_prob),
            "classified_by": "LowConfidence"
        }

    return {
        "log_level": predicted_level,
        "confidence": float(max_prob),
        "classified_by": "BERT-ML"
    }

# -------------------------------
# Example
# -------------------------------
example_log = "error while retriving"

result = predict_log_level(example_log)
print(result)


{'log_level': np.str_('INFO'), 'confidence': 0.8005935903414354, 'classified_by': 'BERT-ML'}


In [ ]:
df["log_level"].value_counts()

,count
log_level,
INFO,5693
NOTICE,1405
ERROR,1179
UNKNOWN,250
WARN,143


In [ ]:
df[df["log_level"]=="UNKNOWN"]

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,level,confidence,classified_by
35,mac,Jul 1 09:29:02 calvisitor-10-105-160-95 sandb...,jul <NUM> <NUM>:<NUM>:<NUM> calvisitor-<NUM>-<...,UNKNOWN,macos,NaT,{},NaN,None,0.0,Unclassified
86,mac,Jul 1 10:38:53 calvisitor-10-105-160-95 sandb...,jul <NUM> <NUM>:<NUM>:<NUM> calvisitor-<NUM>-<...,UNKNOWN,macos,NaT,{},NaN,None,0.0,Unclassified
90,mac,Jul 1 10:47:08 calvisitor-10-105-160-95 sandb...,jul <NUM> <NUM>:<NUM>:<NUM> calvisitor-<NUM>-<...,UNKNOWN,macos,NaT,{},NaN,None,0.0,Unclassified
154,mac,Jul 1 13:20:33 calvisitor-10-105-160-95 sandb...,jul <NUM> <NUM>:<NUM>:<NUM> calvisitor-<NUM>-<...,UNKNOWN,macos,NaT,{},NaN,None,0.0,Unclassified
169,mac,Jul 1 14:29:01 calvisitor-10-105-160-95 sandb...,jul <NUM> <NUM>:<NUM>:<NUM> calvisitor-<NUM>-<...,UNKNOWN,macos,NaT,{},NaN,None,0.0,Unclassified
...,...,...,...,...,...,...,...,...,...,...,...
7739,bgl,KERNTERM 1131680322 2005.11.10 R63-M0-N6-C:J12...,kernterm <NUM> <NUM>.<NUM>.<NUM> r63-m0-n6-c:j...,UNKNOWN,bgl,NaT,{},NaN,None,0.0,Unclassified
7747,bgl,KERNTERM 1132021523 2005.11.14 R46-M1-NE-C:J14...,kernterm <NUM> <NUM>.<NUM>.<NUM> r46-m1-ne-c:j...,UNKNOWN,bgl,NaT,{},NaN,None,0.0,Unclassified
7756,bgl,KERNTERM 1132111168 2005.11.15 R57-M0-NA-C:J13...,kernterm <NUM> <NUM>.<NUM>.<NUM> r57-m0-na-c:j...,UNKNOWN,bgl,NaT,{},NaN,None,0.0,Unclassified
7767,bgl,KERNMNTF 1132575798 2005.11.21 R04-M1-N8-I:J18...,kernmntf <NUM> <NUM>.<NUM>.<NUM> r04-m1-n8-i:j...,UNKNOWN,bgl,NaT,{},NaN,None,0.0,Unclassified


In [2]:
import pandas as pd
import numpy as np
import torch
import joblib
import os

from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler

os.makedirs("model", exist_ok=True)

# ----------------------------------
# 1. Load data
# ----------------------------------
df = pd.read_json("regex_output.ndjson",lines=True)

# ----------------------------------
# 2. Select training data ONLY
# ----------------------------------
df = df[
    (df["classified_by"] == "Unclassified") &
    (df["log_level"].notna())
]

# OPTIONAL but recommended
df = df[df["log_level"] != "UNKNOWN"]

print("Class distribution before fix:")
print(df["log_level"].value_counts())

texts = df["clean_message"].astype(str).tolist()
labels = df["log_level"].tolist()

# ----------------------------------
# 3. Encode labels
# ----------------------------------
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
joblib.dump(label_encoder, "model/label_encoder.pkl")

# ----------------------------------
# 4. Load BERT
# ----------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()

# ----------------------------------
# 5. BERT embeddings
# ----------------------------------
def bert_embeddings(texts, batch_size=16):
    vectors = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            )
            outputs = bert(**inputs)
            cls = outputs.last_hidden_state[:, 0, :]
            vectors.append(cls.cpu().numpy())
    return np.vstack(vectors)

X = bert_embeddings(texts)

# ----------------------------------
# 6. Train / Test split (STRATIFIED)
# ----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ----------------------------------
# 7. Oversample minority classes (TRAIN ONLY)
# ----------------------------------
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

print("Class distribution after oversampling:")
print(pd.Series(y_train_bal).value_counts())

# ----------------------------------
# 8. Train classifier
# ----------------------------------
clf = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    class_weight="balanced"
)

clf.fit(X_train_bal, y_train_bal)

# ----------------------------------
# 9. Evaluation
# ----------------------------------
y_pred = clf.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

# ----------------------------------
# 10. Save model
# ----------------------------------
joblib.dump(clf, "model/log_level_clf.pkl")

print("✅ Phase-2 trained with imbalance handling")


Class distribution before fix:
log_level
INFO      5693
NOTICE    1405
ERROR     1179
WARN       143
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Class distribution after oversampling:
2    4554
1    4554
0    4554
3    4554
Name: count, dtype: int64
              precision    recall  f1-score   support

       ERROR       0.97      0.98      0.98       236
        INFO       1.00      0.99      1.00      1139
      NOTICE       1.00      1.00      1.00       281
        WARN       1.00      1.00      1.00        28

    accuracy                           0.99      1684
   macro avg       0.99      0.99      0.99      1684
weighted avg       0.99      0.99      0.99      1684

✅ Phase-2 trained with imbalance handling


In [13]:
import torch
import joblib
import numpy as np
from transformers import BertTokenizer, BertModel

# -------------------------------
# Load models
# -------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()

clf = joblib.load("model/log_level_clf.pkl")
label_encoder = joblib.load("model/label_encoder.pkl")

# -------------------------------
# BERT embedding function
# -------------------------------
def embed(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert(**inputs)

    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

# -------------------------------
# Predict log level
# -------------------------------
def predict_log_level(log_message, threshold=0.6):
    vector = embed(log_message)

    probs = clf.predict_proba(vector)[0]
    max_prob = np.max(probs)
    label_idx = np.argmax(probs)

    predicted_level = label_encoder.inverse_transform([label_idx])[0]

    if max_prob < threshold:
        return {
            "log_level": "Uncertain",
            "confidence": float(max_prob),
            "classified_by": "LowConfidence"
        }

    return {
        "log_level": predicted_level,
        "confidence": float(max_prob),
        "classified_by": "BERT-ML"
    }

# -------------------------------
# Example
# -------------------------------
example_log = "disk space running low"

result = predict_log_level(example_log)
print(result)


{'log_level': np.str_('INFO'), 'confidence': 0.999206014971501, 'classified_by': 'BERT-ML'}


In [14]:
import pandas as pd
import json
from sklearn.model_selection import GroupShuffleSplit

# 1. Load your NDJSON data
data = []
with open('regex_output.ndjson', 'r') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

# 2. Select the Input Feature
# CRITICAL: Use 'clean_message' (variables masked), NOT 'raw_log'.
# 'raw_log' contains unique IDs (IPs, timestamps) that cause memorization.
X = df['clean_message']
y = df['log_level']  # Or your specific target label column

# 3. The "Anti-Leakage" Split
# We group by 'clean_message'. This ensures all instances of a specific
# log template go EITHER to train OR to test, never both.
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df['clean_message']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

# 4. Verify the Split
print(f"Total Unique Templates: {df['clean_message'].nunique()}")
print(f"Train Templates: {train_df['clean_message'].nunique()}")
print(f"Test Templates: {test_df['clean_message'].nunique()}")

# Check for leakage (Should be 0)
common_templates = set(train_df['clean_message']).intersection(set(test_df['clean_message']))
print(f"Leakage (Common Templates in Train & Test): {len(common_templates)}")
# If this is > 0, your evaluation metrics are lying to you.

Total Unique Templates: 2403
Train Templates: 1922
Test Templates: 481
Leakage (Common Templates in Train & Test): 0


In [15]:
WARN_PATTERNS = [
    "disk space running low",
    "memory usage high",
    "cpu usage high",
    "approaching limit",
    "retrying",
    "timeout",
    "slow response",
    "degraded",
    "fallback"
]

def semantic_precheck(log):
    text = log.lower()
    for p in WARN_PATTERNS:
        if p in text:
            return {
                "log_level": "WARN",
                "confidence": 1.0,
                "classified_by": "SemanticRule"
            }
    return None

def predict_log_level(log_message):
    # ---- Phase-1a: Semantic rule ----
    rule_result = semantic_precheck(log_message)
    if rule_result:
        return rule_result

    # ---- Phase-2: ML ----
    vector = embed(log_message)
    probs = clf.predict_proba(vector)[0]

    label_idx = np.argmax(probs)
    max_prob = probs[label_idx]
    predicted_level = label_encoder.inverse_transform([label_idx])[0]

    if max_prob < 0.6:
        return {
            "log_level": "Uncertain",
            "confidence": float(max_prob),
            "classified_by": "LowConfidence"
        }

    return {
        "log_level": predicted_level,
        "confidence": float(max_prob),
        "classified_by": "BERT-ML"
    }


In [16]:
import re

def normalize_log(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [17]:
def embed(text):
    text = normalize_log(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert(**inputs)

    return outputs.last_hidden_state[:, 0, :].cpu().numpy()


In [18]:
CLASS_THRESHOLDS = {
    "INFO": 0.92,
    "NOTICE": 0.75,
    "WARN": 0.70,
    "ERROR": 0.65
}


In [19]:
def predict_log_level(log_message):
    vector = embed(log_message)

    probs = clf.predict_proba(vector)[0]
    label_idx = np.argmax(probs)
    max_prob = probs[label_idx]

    predicted_level = label_encoder.inverse_transform([label_idx])[0]
    threshold = CLASS_THRESHOLDS.get(predicted_level, 0.75)

    if max_prob < threshold:
        return {
            "log_level": "Uncertain",
            "confidence": float(max_prob),
            "classified_by": "LowConfidence"
        }

    return {
        "log_level": predicted_level,
        "confidence": float(max_prob),
        "classified_by": "BERT-ML"
    }


In [20]:
ERROR_CUES = [
    "failed", "exception", "traceback", "crash",
    "unable to", "cannot", "terminated", "panic"
]


In [22]:
import re
import torch
import joblib
import numpy as np
from transformers import BertTokenizer, BertModel

# =========================================================
# 1. LOAD MODELS
# =========================================================

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()

clf = joblib.load("model/log_level_clf.pkl")
label_encoder = joblib.load("model/label_encoder.pkl")

# =========================================================
# 2. NORMALIZATION (MANDATORY)
# =========================================================

def normalize_log(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# =========================================================
# 3. SEMANTIC RULES (BEFORE ML)
# =========================================================

ERROR_CUES = [
    "failed", "exception", "traceback",
    "crash", "panic", "segfault",
    "unable to", "cannot", "terminated"
]

WARN_CUES = [
    "disk space running low",
    "memory usage high",
    "cpu usage high",
    "approaching limit",
    "retrying",
    "timeout",
    "degraded",
    "fallback"
]

def semantic_precheck(text: str):
    t = text.lower()

    for cue in ERROR_CUES:
        if cue in t:
            return {
                "log_level": "ERROR",
                "confidence": 1.0,
                "classified_by": "SemanticRule-ERROR"
            }

    for cue in WARN_CUES:
        if cue in t:
            return {
                "log_level": "WARN",
                "confidence": 1.0,
                "classified_by": "SemanticRule-WARN"
            }

    return None

# =========================================================
# 4. BERT EMBEDDING
# =========================================================

def embed(text: str) -> np.ndarray:
    text = normalize_log(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert(**inputs)

    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

# =========================================================
# 5. CLASS-AWARE THRESHOLDS
# =========================================================

CLASS_THRESHOLDS = {
    "INFO": 0.92,
    "NOTICE": 0.75,
    "ERROR": 0.65
}

# =========================================================
# 6. FINAL PREDICTION LOGIC
# =========================================================

def predict_log_level(log_message: str):
    """
    This function is called ONLY if:
    - regex did not classify the log
    - classified_by == 'Unclassified'
    """

    # ---- Step 1: Semantic rules ----
    rule_result = semantic_precheck(log_message)
    if rule_result:
        return rule_result

    # ---- Step 2: ML prediction ----
    vector = embed(log_message)

    probs = clf.predict_proba(vector)[0]
    label_idx = np.argmax(probs)
    max_prob = probs[label_idx]

    predicted_level = label_encoder.inverse_transform([label_idx])[0]
    threshold = CLASS_THRESHOLDS.get(predicted_level, 0.75)

    normalized = normalize_log(log_message)

    # ---- Step 3: Semantic ERROR override ----
    if predicted_level == "INFO":
        if any(cue in normalized for cue in ERROR_CUES):
            return {
                "log_level": "Uncertain",
                "confidence": float(max_prob),
                "classified_by": "SemanticOverride-ERROR"
            }

    # ---- Step 4: Confidence gate ----
    if max_prob < threshold:
        return {
            "log_level": "Uncertain",
            "confidence": float(max_prob),
            "classified_by": "LowConfidence"
        }

    # ---- Step 5: Accept ML decision ----
    return {
        "log_level": predicted_level,
        "confidence": float(max_prob),
        "classified_by": "BERT-ML"
    }

# =========================================================
# 7. EXAMPLES
# =========================================================

if __name__ == "__main__":
    tests = [
        "error while retriving",
        "failed to connect to database",
        "disk space running low",
        "service started successfully",
        "retrying request after timeout"
        "Issue while retriving"
    ]

    for t in tests:
        print(t)
        print(predict_log_level(t))
        print("-" * 50)


error while retriving
{'log_level': 'Uncertain', 'confidence': 0.9096948306710687, 'classified_by': 'LowConfidence'}
--------------------------------------------------
failed to connect to database
{'log_level': 'ERROR', 'confidence': 1.0, 'classified_by': 'SemanticRule-ERROR'}
--------------------------------------------------
disk space running low
{'log_level': 'WARN', 'confidence': 1.0, 'classified_by': 'SemanticRule-WARN'}
--------------------------------------------------
service started successfully
{'log_level': np.str_('INFO'), 'confidence': 0.9930803048561178, 'classified_by': 'BERT-ML'}
--------------------------------------------------
retrying request after timeoutIssue while retriving
{'log_level': 'WARN', 'confidence': 1.0, 'classified_by': 'SemanticRule-WARN'}
--------------------------------------------------
